# **Thames Valley Crime Data Analysis**
## ETL: Data Preparation and Cleaning
---

### Objectives

* Combine the Thames Valley Police street-level crime data (October 2023 to September 2026) into one dataset, inspect it, clean it, and summarise it for data analysis, modelling and the dashboard.

* Check for missing values, duplicated and unneeded columns, and explain how each are handled and why.

* Create an **Area** column, so crimes can be compared by area.

* Create a **Season** column so that crimes can be compared by season (spring, summer, autumn, winter)

* Save a summary of crimes by crime counts by **Month**, **Area** and **Crime Type**

* Work out the average of **Latitude** and **Longitude** to draw a map with a bubble per area.

### Planned Steps

1. Load and combine the raw data

2. Inspection of data

3. Conclusion of data inspection and create a cleaning strategy

4. Identify columns required for analysis

5. Clean data

6. Summarise data and validate

7. Export summarised datasets for analysis, visualisation, modelling and dashboard

### Inputs

**Data Source:** [Thames Valley Police street-level crime data](https://data.police.uk/data/)

Raw data is comprised of 36 monthly CSV files saved in `../data/raw`

### Outputs

* Full clean dataset: `../data/clean/crime_clean.csv`

* Summarised crime counts by month, season, area, and crime type - **Exported:** `../data/clean/crime_summary.csv`

* Centre point of each area (average latitude and longitude) - **Exported:** `../data/clean/area_locations.csv`

### Additional Comments

* The raw data files will never be changed.

* All cleaning takes place in this notebook, so the steps can be checked for problems or repeated.

* Cells in this notebook run top to bottom
---

### Setup

Install and import required libraries

In [1]:
# Installs pandas
%pip install pandas

# Imports required libraries
import os
import pandas as pd

print("Libraries imported Successfully")

Note: you may need to restart the kernel to use updated packages.
Libraries imported Successfully


---
### 1. Load and combine the raw data

In [2]:
# List the total of files in the raw data folder
files = os.listdir("../data/raw")
print("Number of Files:", len(files))

Number of Files: 36


In [3]:
# Load every file in the raw data folder and add it to a list
all_months = []
for file in files:
    month_df = pd.read_csv("../data/raw/" + file)
    all_months.append(month_df)

# Join all the months into one dataset
df = pd.concat(all_months, ignore_index=True)
print("Total Rows:", len(df))

Total Rows: 592926


In [4]:
# Record the size of the raw data, to compare with the cleaned data
raw_rows, raw_cols = df.shape
raw_file_mb = sum(os.path.getsize("../data/raw/" + file) for file in files) / (1024 * 1024)
raw_memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)

print(f"Raw data: {raw_rows} rows, {raw_cols} columns")
print(f"File size: {raw_file_mb:.1f} MB")
print(f"Memory usage: {raw_memory_mb:.1f} MB")

Raw data: 592926 rows, 12 columns
File size: 131.9 MB
Memory usage: 375.1 MB


---
### 2. Inspection of data

In [5]:
# Number of rows and columns
df.shape

(592926, 12)

In [6]:
# Show the first 5 rows, so we can see the columns and some values
df.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,17c7dba97ddd4c8ce057762eed5ca44a5e276d59de8183...,2023-10,Thames Valley Police,Thames Valley Police,-0.680255,51.450425,On or near Crouch Lane,E01016252,Bracknell Forest 001C,Burglary,Investigation complete; no suspect identified,NaN
1,acd686143e4a14096d6cf482f1cc86adc4bad30048b3b8...,2023-10,Thames Valley Police,Thames Valley Police,-0.692549,51.452793,On or near Winkfield Lane,E01016252,Bracknell Forest 001C,Violence and sexual offences,Unable to prosecute suspect,NaN
2,8ba67cb1b6302cf07022cb009f8ac3a6d718c40d9102bc...,2023-10,Thames Valley Police,Thames Valley Police,-0.667222,51.447760,On or near Homelands,E01016252,Bracknell Forest 001C,Violence and sexual offences,Status update unavailable,NaN
3,237aa62135742fc3bdd2be4837b5c2b6d8039ac8e39842...,2023-10,Thames Valley Police,Thames Valley Police,-0.718967,51.433967,On or near Cricketers Lane,E01035234,Bracknell Forest 001F,Vehicle crime,Court result unavailable,NaN
4,b74bea25ecedac56f84a434912430c1588ef813c9fb64e...,2023-10,Thames Valley Police,Thames Valley Police,-0.710564,51.445268,On or near Winkfield Lane,E01035234,Bracknell Forest 001F,Violence and sexual offences,Unable to prosecute suspect,NaN


In [7]:
# Show the data type of each column
df.dtypes

Crime ID                  object
Month                     object
Reported by               object
Falls within              object
Longitude                float64
Latitude                 float64
Location                  object
LSOA code                 object
LSOA name                 object
Crime type                object
Last outcome category     object
Context                  float64
dtype: object

In [8]:
# Number of null values in each column
df.isnull().sum()

Crime ID                  61493
Month                         0
Reported by                   0
Falls within                  0
Longitude                 13132
Latitude                  13132
Location                      0
LSOA code                 13132
LSOA name                 13132
Crime type                    0
Last outcome category     61493
Context                  592926
dtype: int64

In [9]:
# Percentage of null values in each column
df.isnull().sum() / len(df) * 100

Crime ID                  10.371109
Month                      0.000000
Reported by                0.000000
Falls within               0.000000
Longitude                  2.214779
Latitude                   2.214779
Location                   0.000000
LSOA code                  2.214779
LSOA name                  2.214779
Crime type                 0.000000
Last outcome category     10.371109
Context                  100.000000
dtype: float64

In [10]:
# Summary statistics for all columns
df.describe(include="all")

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
count,531433,592926,592926,592926,579794.000000,579794.000000,592926,579794,579794,592926,531433,0.0
unique,531239,36,1,1,NaN,NaN,21167,1491,1491,14,15,NaN
top,927e9abd7503e9ef10c42eb2da70b7b3809b51d4e0983f...,2026-07,Thames Valley Police,Thames Valley Police,NaN,NaN,On or near Supermarket,E01033415,Reading 019B,Violence and sexual offences,Unable to prosecute suspect,NaN
freq,5,19063,592926,592926,NaN,NaN,30152,8990,8990,223827,200810,NaN
mean,NaN,NaN,NaN,NaN,-0.930321,51.678288,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,0.271025,0.224866,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,-1.685051,51.336336,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,-1.199666,51.475339,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,-0.828079,51.630422,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,-0.731599,51.824414,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Look at the rows with no location
missing_rows = df[df["LSOA name"].isnull()]
missing_rows.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
16807,NaN,2023-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Anti-social behaviour,NaN,NaN
16808,NaN,2023-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Anti-social behaviour,NaN,NaN
16809,NaN,2023-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Anti-social behaviour,NaN,NaN
16810,NaN,2023-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Anti-social behaviour,NaN,NaN
16811,NaN,2023-10,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Anti-social behaviour,NaN,NaN


In [12]:
# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 11424


In [13]:
duplicates = df[df.duplicated()]
duplicates

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
107,NaN,2023-10,Thames Valley Police,Thames Valley Police,-0.761271,51.420871,On or near Arlington Close,E01016240,Bracknell Forest 004B,Anti-social behaviour,NaN,NaN
256,NaN,2023-10,Thames Valley Police,Thames Valley Police,-0.752138,51.405577,On or near Chadwick Mews,E01016233,Bracknell Forest 007B,Anti-social behaviour,NaN,NaN
257,NaN,2023-10,Thames Valley Police,Thames Valley Police,-0.752138,51.405577,On or near Chadwick Mews,E01016233,Bracknell Forest 007B,Anti-social behaviour,NaN,NaN
292,NaN,2023-10,Thames Valley Police,Thames Valley Police,-0.758035,51.407097,On or near Wyvern Close,E01016250,Bracknell Forest 007E,Anti-social behaviour,NaN,NaN
315,NaN,2023-10,Thames Valley Police,Thames Valley Police,-0.746457,51.417746,On or near Weather Way,E01035241,Bracknell Forest 007F,Anti-social behaviour,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
592782,a883918c8c683dedb3af84ab6f16f45c778f3a4c4c75c8...,2026-09,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation,NaN
592820,faa1a9e30595bc83d5c37cdb50d6957a0022adbb1986db...,2026-09,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation,NaN
592867,53c8181409ed4e5f3219b066d51f8ef77f1c59a370fee9...,2026-09,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation,NaN
592897,231d15759180d26fa422f5de470af219796ed41a481906...,2026-09,Thames Valley Police,Thames Valley Police,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Under investigation,NaN


In [14]:
# How many times does each repeated Crime ID appear
id_counts = df['Crime ID'].value_counts()
id_counts[id_counts > 1]

Crime ID
927e9abd7503e9ef10c42eb2da70b7b3809b51d4e0983faed03ebddf081d069b    5
b96167c2dc1d9d2efe349732377cbc7f6b311f5c92a02c77d654bb9084c257c9    4
5a3687f3b3772dda70b40072e4955fc4fd0184c9b486fad280ed6055489d19d7    4
00167d3574e363b8fb0596a2b0d8b1f0434196b9ab5b67ad4468610e3ded9bc9    4
d63274543dc3d831d97355264a7305a87c34e1bfd4659a30be04072794f07415    3
                                                                   ..
2eb5eedd8972707d22bb73f8aa40e73fdb535310904a3f21ee73bfe8f59cc3d9    2
386b53211b212fda578046f6462ce3951cd5aef83a1766c1b607f7f797ac8e2b    2
119d7975013a79009df366082d48420c7a044aa7233a6bcf98eb02c4f961e1d9    2
3fa85f97011abb82609b29e4a9341f089a5f772bf418ccce8fd6f261885a717e    2
2e04119714018b6662c14069c8597dd5ba3979b97fc568ea2aca85b8c8ba6666    2
Name: count, Length: 163, dtype: int64

In [15]:
# Summary of repeated crime IDs
id_counts = df["Crime ID"].value_counts()
repeated_ids = (id_counts > 1).sum()
extra_rows = df["Crime ID"].count() - df["Crime ID"].nunique()

print(f"Total number of rows in the dataset: {len(df)}")
print(f"Number of unique crime IDs: {df['Crime ID'].nunique()}")
print(f"Number of repeated crime IDs: {repeated_ids}")
print(f"Number of extra rows caused by repeated crime IDs: {extra_rows}")
print(f"Percentage of extra rows caused by repeated crime IDs: {extra_rows / len(df) * 100:.2f}%")

Total number of rows in the dataset: 592926
Number of unique crime IDs: 531239
Number of repeated crime IDs: 163
Number of extra rows caused by repeated crime IDs: 194
Percentage of extra rows caused by repeated crime IDs: 0.03%


In [16]:
# Check for missing crime type
print("Missing crime type:", df["Crime type"].isnull().sum())

# How many crime types there are?
print("Number of crime types:", df["Crime type"].nunique())

# Show the total count of all crime types
df["Crime type"].value_counts()

Missing crime type: 0
Number of crime types: 14


Crime type
Violence and sexual offences    223827
Anti-social behaviour            61493
Shoplifting                      58015
Other theft                      46716
Public order                     45693
Criminal damage and arson        43235
Vehicle crime                    35960
Burglary                         21530
Drugs                            18045
Other crime                      11887
Bicycle theft                     9546
Theft from the person             6829
Robbery                           5994
Possession of weapons             4156
Name: count, dtype: int64

---
### 3. Conclusion of data inspection and create a cleaning strategy

What we found out from the data inspection:-
- The combined dataset has 592,926 rows and 12 columns.
- ***Context*** is completely empty (100% missing).
- ***Crime ID*** is missing for anti-social behaviour, which has no ID; this occurs on 61,493 rows.
- ***Crime ID*** has 163 IDs that appear more than once, giving 194 extra rows (0.03%). Most repeat twice, and the highest repeats 5 times.
- ***LSOA code*** and ***LSOA name*** are missing from 13,132 rows (2.21%), so those crimes can not be put into an area.
- ***Reported by*** and ***Falls within*** have the same value on every row.
- There are 11,424 duplicate rows.
- ***Crime type*** has no missing values and 14 different crime types.

Cleaning Strategy:-
- Keep only the columns needed for the key questions: Month, LSOA name, Crime type, and latitude and longitude to work out the centre of each area for a map.
- Remove the rows with no LSOA, as they can not be placed in an area.
- Keep the 194 repeated **Crime ID** rows, at 0.03% of the data, they are too few to have any significant effect on the data.
- Create ***Area*** column from the ***LSOA name***, and keep only the 13 Thames Valley areas.
- Convert ***Month*** from text to a date, and add a ***Season*** column.
- Keep duplicate rows: Identical rows can be two real crimes, because locations are rounded for privacy and anti-social behaviour has no ***Crime ID***.

---
### 4. Identify columns required for analysis

The key questions we are looking to answer are about *when* (**Month**), *where* (**Area**), and *what* (**Crime type**).

Columns Kept:
- ***Month***: to compare crimes over time and create the ***Season*** column.
- ***LSOA name***: used to create the ***Area*** column.
- ***Crime type***: to compare types of crime.
- ***Latitude*** and ***Longitude***: kept for now to work out the centre of each area for a map.

Columns Dropped:
- ***Crime ID***: only needed to check for repeated crimes, which was done in Section 2 (above).
- ***Reported by*** and ***Falls within***: the same value on every row, so they tell us nothing.
- ***Location***: street-level data is not needed, as crimes are being compared by area.
- ***LSOA code***: could be used to add population data and compare crime rates per 1,000 people, which would be fairer than raw counts. This is outside the scope of this project, so it has been removed.
- ***Last outcome category***: crime outcomes are not required to answer any part of our key questions.
- ***Context***: Column is empty throughout.

---
### 5. Clean Data

The first job is to remove the columns that aren't required from the dataframe. Then check what we have left.

In [17]:
# Drop the columns not required for analysis 
df=df.drop(columns=["Crime ID", "Reported by", "Falls within", "Location", "LSOA code", "Last outcome category", "Context"])

df.head()

,Month,Longitude,Latitude,LSOA name,Crime type
0,2023-10,-0.680255,51.450425,Bracknell Forest 001C,Burglary
1,2023-10,-0.692549,51.452793,Bracknell Forest 001C,Violence and sexual offences
2,2023-10,-0.667222,51.447760,Bracknell Forest 001C,Violence and sexual offences
3,2023-10,-0.718967,51.433967,Bracknell Forest 001F,Vehicle crime
4,2023-10,-0.710564,51.445268,Bracknell Forest 001F,Violence and sexual offences


#### Missing Values

Before we remove any rows, we need to check how many rows are affected and whether removing them could change or influence the answers to our key questions.

In [18]:
# Missing values in the columns we kept
df.isnull().sum()

Month             0
Longitude     13132
Latitude      13132
LSOA name     13132
Crime type        0
dtype: int64

In [19]:
# Percentage of rows with no location
print("Rows with no LSOA:", df["LSOA name"].isnull().sum())
print("Percentage of all rows:", round(df["LSOA name"].isnull().sum() / len(df) * 100, 2))

Rows with no LSOA: 13132
Percentage of all rows: 2.21


This returns 13,132 rows with no Crime Location, which makes up 2.21% of the data. With such a small percentage given the overall size of the dataset, it is unlikely to change the result significantly.

In [20]:
# Compare the crime types in the missing rows, with all rows
comparison = pd.DataFrame({
    "All rows (%)": df["Crime type"].value_counts(normalize=True) * 100,
    "Rows with no location (%)": df[df["LSOA name"].isnull()]["Crime type"].value_counts(normalize=True) * 100,
}).fillna(0).round(1)
comparison

,All rows (%),Rows with no location (%)
Crime type,,
Anti-social behaviour,10.4,6.8
Bicycle theft,1.6,1.1
Burglary,3.6,2.6
Criminal damage and arson,7.3,5.7
Drugs,3.0,7.0
Other crime,2.0,3.0
Other theft,7.9,7.2
Possession of weapons,0.7,0.8
Public order,7.7,6.1


- `value_counts(normalize=True)` gives each crime type a share of the rows. Multiplying them by 100 makes them a percentage.

- We are making both columns percentages so they can be compared.

- `fillna(0)` puts in a 0 where a crime type doesn't appear in the rows with no location, instead of leaving it empty.

<br>

**Considerations regarding missing values**:

- **Crime type** and **Month** have no missing values, so nothing needs removing from them.

- 13,132 rows (2.21%) have no **LSOA**, so they can't be placed in an area.

- The crime type mix in these rows is mostly similar to the rest of the data.

    - **Violence and sexual offences** (44.7% vs 37.7%) and **Drugs** (7.0% vs 3.0%) make up a bigger share of the rows with no location.

    - **Anti-social behaviour** makes up a smaller share (6.8% vs 10.4%).
    
- Because these rows are only 2.21% of the data, removing them does not change which crime types are most common.

- **Violence and sexual offences** and **Drugs** will be slightly undercounted in the area results, which will be noted in the limitations.

- Filling in a guessed area would be less accurate than removing these rows, because there is no reliable way to know where the crime happened.
<br>
<br>

**Final Decision:** Remove the 13,132 rows with no **LSOA name**.

In [21]:
# Remove rows with no location
df = df.dropna(subset=["LSOA name"])
print("Rows left after removing those with no location:", len(df))

# Reset the index after dropping rows
df = df.reset_index(drop=True)

df.isnull().sum()

Rows left after removing those with no location: 579794


Month         0
Longitude     0
Latitude      0
LSOA name     0
Crime type    0
dtype: int64

#### Create an Area Column

An LSOA name looks like "Bracknell Forest 001C". The area is everything except the last 5 characters (a space and a 4-character code), so we remove these to leave the area name.

In [22]:
df["Area"] = df["LSOA name"].str.slice(stop=-5)
df["Area"].value_counts()

Area
Buckinghamshire           107913
Milton Keynes              90251
Reading                    59777
Oxford                     54340
Slough                     50706
Cherwell                   36832
West Berkshire             32641
Windsor and Maidenhead     31291
Bracknell Forest           26195
Wokingham                  25956
Vale of White Horse        23925
South Oxfordshire          22319
West Oxfordshire           17626
Central Bedfordshire           8
Dacorum                        7
Basingstoke and Deane          7
Name: count, dtype: int64

In [23]:
# Check the head of the dataframe to ensure the Area column has been created
df.head()

,Month,Longitude,Latitude,LSOA name,Crime type,Area
0,2023-10,-0.680255,51.450425,Bracknell Forest 001C,Burglary,Bracknell Forest
1,2023-10,-0.692549,51.452793,Bracknell Forest 001C,Violence and sexual offences,Bracknell Forest
2,2023-10,-0.667222,51.447760,Bracknell Forest 001C,Violence and sexual offences,Bracknell Forest
3,2023-10,-0.718967,51.433967,Bracknell Forest 001F,Vehicle crime,Bracknell Forest
4,2023-10,-0.710564,51.445268,Bracknell Forest 001F,Violence and sexual offences,Bracknell Forest


#### Keep only the 13 Thames Valley areas

The Thames Valley Police area is made up of [13 local authorities](https://www.westberks.gov.uk/article/42710/Police-Area-Returning-Officer-PARO-2024).

The Area column shows 22 crimes recorded just outside these areas (Central Bedfordshire, Dacorum, and Basingstoke and Deane).

These will be removed.

In [24]:
# Keep only the 13 official Thames Valley areas
thames_valley_areas = [
    "Bracknell Forest", "Buckinghamshire", "Cherwell", "Milton Keynes",
    "Oxford", "Reading", "Slough", "South Oxfordshire", "Vale of White Horse",
    "West Berkshire", "West Oxfordshire", "Windsor and Maidenhead", "Wokingham",
]

df = df[df["Area"].isin(thames_valley_areas)]
df = df.reset_index(drop=True)

print("Areas:", df["Area"].nunique())
print("Rows left:", len(df))

Areas: 13
Rows left: 579772


In [25]:
# Check only the 13 official areas are left
sorted(df["Area"].unique())

['Bracknell Forest',
 'Buckinghamshire',
 'Cherwell',
 'Milton Keynes',
 'Oxford',
 'Reading',
 'Slough',
 'South Oxfordshire',
 'Vale of White Horse',
 'West Berkshire',
 'West Oxfordshire',
 'Windsor and Maidenhead',
 'Wokingham']

#### Work out the centre of each area for a map

To create a bubble map, we work out the centre point of each area by taking the average ***Latitude*** and ***Longitude*** of all the crimes in that area. The results are saved in a new table called `area_locations`, with one row per area, which will be used to draw the bubble map in the dashboard.

In [26]:
area_locations = df.groupby("Area")[["Latitude", "Longitude"]].mean().round(4).reset_index()
area_locations

,Area,Latitude,Longitude
0,Bracknell Forest,51.4011,-0.7580
1,Buckinghamshire,51.7034,-0.7425
2,Cherwell,51.9777,-1.2734
3,Milton Keynes,52.0353,-0.7523
4,Oxford,51.7467,-1.2338
5,Reading,51.4515,-0.9797
6,Slough,51.5125,-0.5946
7,South Oxfordshire,51.6260,-1.1021
8,Vale of White Horse,51.6476,-1.3719
9,West Berkshire,51.4170,-1.2473


#### Convert Month to a date

***Month*** is currently stored as text. Converting it to datetime means:
- the month number can be used to create the ***Season*** column.
- charts treat it as a time axis, in the correct order.
- it is easier to filter by date in the dashboard.

Pandas sets each date to the 1st of the month, for example (2023-10-01), which for our purpose stands for October 2023.

In [27]:
# Convert Month from text to datetime
df["Month"] = pd.to_datetime(df["Month"])

print("Data type of 'Month' column:", df["Month"].dtype)
print("First month:", df["Month"].min())
print("Last month:", df["Month"].max())
print("Number of months:", df["Month"].nunique())

Data type of 'Month' column: datetime64[ns]
First month: 2023-10-01 00:00:00
Last month: 2026-09-01 00:00:00
Number of months: 36


#### Add a Season column

Each month will be assigned a season, so seasonal patterns can be compared in the analysis and the dashboard:
- **Winter** - December, January, February
- **Spring** - March, April, May
- **Summer** - June, July, August
- **Autumn** - September, October, November

The data covers 36 months, so each of the 4 seasons has exactly 9 months of data. This means that the seasons can be compared fairly.

In [28]:
# Link each month number to a season
seasons = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 
           4: "Spring", 5: "Spring", 6: "Summer", 7: "Summer", 
           8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"
           }

# Create a season column from the month number
df["Season"] = df["Month"].dt.month.map(seasons)

# Check each season has 9 months
df.groupby("Season")["Month"].nunique()


Season
Autumn    9
Spring    9
Summer    9
Winter    9
Name: Month, dtype: int64

---
### 6. Summarise data and validate

The cleaned data has one row per crime. For the analysis, modelling and dashboard we only need the **number** of crimes, so the data is summarised into a count of crimes for each month, area and crime type.

This will be done in 2 steps:

1. **Pivot table**: One row for each month and area, with one column for each crime type. Where an area has no crime of a type in a month, a 0 is filled in.

In [29]:
# Count crimes for each month and area, with one column per crime type
summary = df.pivot_table(index=["Month", "Season", "Area"], columns="Crime type", aggfunc="size", fill_value=0)

print("Rows:", len(summary), "Crime type columns:", summary.shape[1])

# Shows every area for a month
summary.head(13)

Rows: 468 Crime type columns: 14


Crime type                                Anti-social behaviour  \
Month      Season Area                                            
2023-10-01 Autumn Bracknell Forest                           81   
                  Buckinghamshire                           306   
                  Cherwell                                   90   
                  Milton Keynes                             171   
                  Oxford                                    125   
                  Reading                                   180   
                  Slough                                    103   
                  South Oxfordshire                          55   
                  Vale of White Horse                        76   
                  West Berkshire                            112   
                  West Oxfordshire                           67   
                  Windsor and Maidenhead                     66   
                  Wokingham                                  91   

Crime type                                Bicycle theft  Burglary  \
Month      Season Area                                              
2023-10-01 Autumn Bracknell Forest                   19        27   
                  Buckinghamshire                    27       162   
                  Cherwell                            7        25   
                  Milton Keynes                      23       103   
                  Oxford                            148        75   
                  Reading                            66        81   
                  Slough                             32        59   
                  South Oxfordshire                   3        23   
                  Vale of White Horse                 9        20   
                  West Berkshire                     16        50   
                  West Oxfordshire                    6        26   
                  Windsor and Maidenhead             11        39   
                  Wokingham                          30        35   

Crime type                                Criminal damage and arson  Drugs  \
Month      Season Area                                                       
2023-10-01 Autumn Bracknell Forest                               70     12   
                  Buckinghamshire                               247     63   
                  Cherwell                                       78     21   
                  Milton Keynes                                 146     39   
                  Oxford                                         83     55   
                  Reading                                       101     32   
                  Slough                                        155     42   
                  South Oxfordshire                              47     12   
                  Vale of White Horse                            72     33   
                  West Berkshire                                 78     25   
                  West Oxfordshire                               46     15   
                  Windsor and Maidenhead                         69     30   
                  Wokingham                                      77     17   

Crime type                                Other crime  Other theft  \
Month      Season Area                                               
2023-10-01 Autumn Bracknell Forest                 16           46   
                  Buckinghamshire                  54          259   
                  Cherwell                         27          112   
                  Milton Keynes                    42          205   
                  Oxford                           19          158   
                  Reading                          25          186   
                  Slough                           19          125   
                  South Oxfordshire                 9           49   
                  Vale of White Horse              16           47   
                 

2. **Melt**: The crime type columns are turned into rows, giving one row per month, area and crime type, with a ***Count*** column. This shape is easier to filter, chart and model.

In [30]:
# Turn the crime types into rows, with one count column
summary = summary.reset_index()
summary = summary.melt(id_vars=["Month", "Season", "Area"], var_name="Crime type", value_name="Count")

print("Rows:", len(summary), "Columns:", summary.shape[1])
summary.head()

Rows: 6552 Columns: 5


,Month,Season,Area,Crime type,Count
0,2023-10-01,Autumn,Bracknell Forest,Anti-social behaviour,81
1,2023-10-01,Autumn,Buckinghamshire,Anti-social behaviour,306
2,2023-10-01,Autumn,Cherwell,Anti-social behaviour,90
3,2023-10-01,Autumn,Milton Keynes,Anti-social behaviour,171
4,2023-10-01,Autumn,Oxford,Anti-social behaviour,125


#### Validate

Before exporting, we need to check that no crimes were lost or added while summarising:

- The summary should have **6,552 rows** (36 months x 13 areas x 14 crime types).

- The total of the **Count** column should equal the number of the rows in the cleaned data, **579,772**. If they match, every crime has been counted exactly once.

In [31]:
# Check the summary has one row per month, area and crime type
print("Rows in summary:", len(summary))

# Check no crimes were lost or added when summarising
print("Total crimes in summary:", summary['Count'].sum())
print("Crimes in cleaned data:", len(df))

Rows in summary: 6552
Total crimes in summary: 579772
Crimes in cleaned data: 579772


---
### 7. Export summarised datasets for analysis, visualisation, modelling and dashboard

The cleaned data and summarised data will be saved as three .csv files in `data/clean`, ready for the analysis, modelling and dashboard notebooks:

- `crime_clean.csv` - The full cleaned dataset, with one row per crime.
- `crime_summary.csv` - Crime counts per month, season, area and crime type.
- `area_locations.csv` - The centre point of each of the 13 areas for the bubble map.

In [32]:
# Create a clean data folder if it doesn't exist
os.makedirs("../data/clean", exist_ok=True)
print("Clean data folder created or already exists.")

# Save the three cleaned datasets
df.to_csv("../data/clean/crime_clean.csv", index=False)
summary.to_csv("../data/clean/crime_summary.csv", index=False)
area_locations.to_csv("../data/clean/area_locations.csv", index=False)

print("Files saved successfully.")

Clean data folder created or already exists.
Files saved successfully.


#### Check the size of the exported data files

In [33]:
# Check the size of the cleaned data files

for file_name in ["crime_clean.csv", "crime_summary.csv", "area_locations.csv"]:
    file_path = f"../data/clean/{file_name}"
    if os.path.exists(file_path):
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"{file_name}: {size_mb:.1f} MB")
    else:
        print(f"{file_name} does not exist.")

crime_clean.csv: 50.1 MB
crime_summary.csv: 0.3 MB
area_locations.csv: 0.0 MB


The file sizes are ok, though `crime_clean.csv` is just over GitHub's 50 MB warning limit. This would trigger a warning from GitHub and slow down downloading the repository.

To fix this, three more columns can be dropped, because they have done their job (earlier in the notebook):

- ***LSOA name*** was only needed to create the ***Area*** column.
- ***Latitude*** and ***Longitude*** were only needed to create `area_locations` (the centre point of each area) to create the bubble map.

Dropping these columns should bring the file under the 50 MB GitHub limit.

In [34]:
# Note: this cell should only be run once - running again gives a KeyError, as the columns are already gone

# # Drop the columns already used
df = df.drop(columns=["LSOA name", "Latitude", "Longitude"])

# Verify that the columns have been dropped
print(df.columns.tolist())

# Save the smaller dataset (replacing the old version)
df.to_csv("../data/clean/crime_clean.csv", index=False)
print("Smaller dataset saved successfully.")

# Check the size of the updated cleaned data file
size_mb = os.path.getsize("../data/clean/crime_clean.csv") / (1024 * 1024)
print(f"crime_clean.csv: {size_mb:.1f} MB")

['Month', 'Crime type', 'Area', 'Season']
Smaller dataset saved successfully.
crime_clean.csv: 29.1 MB


The dataset `crime_clean.csv` has now been reduced from 50.1 MB to 29.1 MB, bringing it under GitHub's 50 MB limit.

#### Compare raw and cleaned data

The size of the raw data was recorded after initial loading, before any cleaning. Comparing it with the clean data shows how much the cleaning has reduced the dataset.

- **Rows** - Crimes removed (no location, or outside Thames Valley area).
- **Columns** - Columns dropped as not needed.
- **File size** - Space taken on disk (raw CSV vs `crime_clean.csv`).
- **Memory usage** - Space taken when the data is loaded into pandas.

In [35]:
# Compare the raw data with the cleaned data
clean_file_mb = os.path.getsize("../data/clean/crime_clean.csv") / (1024 * 1024)
clean_memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)

print(f"Rows:         {raw_rows} → {len(df)}")
print(f"Columns:      {raw_cols} → {df.shape[1]}")
print(f"File size:    {raw_file_mb:.1f} MB → {clean_file_mb:.1f} MB ({(1 - clean_file_mb / raw_file_mb) * 100:.0f}% smaller)")
print(f"Memory usage: {raw_memory_mb:.1f} MB → {clean_memory_mb:.1f} MB ({(1 - clean_memory_mb / raw_memory_mb) * 100:.0f}% smaller)")

Rows:         592926 → 579772
Columns:      12 → 4
File size:    131.9 MB → 29.1 MB (78% smaller)
Memory usage: 375.1 MB → 106.6 MB (72% smaller)


The cleaned dataset is 78% smaller on disk than the raw data, and uses 106.6 MB in memory instead of 375.1 MB (72% smaller).

This makes using the datasets both faster and more efficient.

---

### Conclusions and Next Steps

**What we did in this notebook:**
- Combined 36 monthly files into one dataset of 592,926 rows and 12 columns.
- Removed 13,132 rows with no location (2.21%), and 22 rows outside the Thames Valley areas.
- Kept 194 rows with repeated crime IDs (0.03%) and the duplicate rows (explained in section 3).
- Kept ***Month***, ***Area*** and ***Crime type***, which we need to answer our key questions.
- Added a ***Season*** column, which we need for our statistical test.
- Saved a cleaned dataset of 579,772 rows, a summary of 6,552 rows, and the centre point of each of the areas for a bubble map.
- Reduced the data from 592,926 rows and 12 columns (131.9 MB on disk, 375.1 MB in memory) to 579,772 rows and 4 columns (29.1 MB on disk, 106.6 MB in memory).

**Limitations** 
- ***Violence and sexual offences*** (44.7% vs 37.7%) and ***Drugs*** (7.0% vs 3.0%) make up a bigger share of the rows with no location, so they are slightly under-counted in the area results.
- ***Anti-social behaviour*** makes up a smaller share of the rows with no location (6.8% vs 10.4%), so it is slightly over-represented compared with other crimes.

**Future Development**
- Add ONS data to investigate crimes per 1,000 people instead of as raw counts. This would give a fairer comparison as larger areas such as Buckinghamshire naturally have more reported crime.

**Next notebook:** 02_Analysis.ipynb